In [1]:
%load_ext autoreload
%autoreload 2

from IPython.display import display
import json
import os
import numpy as np
import torch
import torchvision.utils as vutils
import pandas as pd
import matplotlib.pyplot as plt
from pycocotools.coco import COCO
from matplotlib import font_manager as fm
import matplotlib.colors as mcolors
import matplotlib.markers as mmarkers
import seaborn as sns
from PIL import Image

# from compcats.localizer import localizer
from compcats.datasets.fmri import nsd
from compcats.utils import model_utils, coco_utils
from compcats.datasets.fmri.utils import nsd_ds_utils

import main_analysis as ma
import paths

# Matplotlib settings
import matplotlib as mpl
from matplotlib import rcParams
rcParams.update({'figure.autolayout': False})
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
# Font
font_path = '/home/alishd/.fonts/Avenir.ttc'
avenir_font = fm.FontProperties(fname=font_path)
mpl.rcParams['font.family'] = avenir_font.get_name()
ttc = fm.get_font(font_path)
print("Font name from TTC:", ttc.family_name)
print("Style:", ttc.style_name)

RESULTS_DIR = 'results/exploration/selecting_canonical_category_images'
os.makedirs(RESULTS_DIR, exist_ok=True)
MODELS_DIR = 'models'
# PLOTS_DIR = f'{RESULTS_DIR}/plots/postvss25'
# os.makedirs(PLOTS_DIR, exist_ok=True)

NSD_DIR = paths.NSD_DIR

coco_train_annotations = COCO(paths.COCO_TRAIN_ANNOTATIONS_PATH)
coco_train_captions = COCO(paths.COCO_TRAIN_CAPTIONS_PATH)
coco_val_annotations = COCO(paths.COCO_VAL_ANNOTATIONS_PATH)
coco_val_captions = COCO(paths.COCO_VAL_CAPTIONS_PATH)

# COCO Variables
coco_animal_labels = ['bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe']
coco_person_labels = ['person']
exclusion_labels = ['teddy bear']

Font name from TTC: Avenir
Style: Book
loading annotations into memory...
Done (t=7.16s)
creating index...
index created!
loading annotations into memory...
Done (t=0.50s)
creating index...
index created!
loading annotations into memory...
Done (t=0.29s)
creating index...
index created!
loading annotations into memory...
Done (t=0.02s)
creating index...
index created!


In [ ]:
df_dict = {}
for sbjidx in range(1, 9):
    sdata, images_df = nsd.get_betas(
            NSD_DIR, sbjidx, 'FFA', 7, subset=None, voxels=False
    )
    df_dict[sbjidx] = images_df
    print(sbjidx)
    print(len(images_df[images_df['num_repetitions'] > 0]))
    print(len(images_df[images_df['num_repetitions'] == 3]))

1
10000
10000
2
10000
10000
3
9411
6234
4
9209
5445
5
10000
10000
6
9411
6234
7
10000
10000
8
9209
5445


In [22]:
shared1000_nsdids = nsd_ds_utils.get_subset_nsdids(NSD_DIR, 1000)

sbjsbj_stimoverlap = np.zeros((8, 8))
for i in range(8):
    for j in range(8):
        s1_df = df_dict[j+1]
        s1_df = s1_df[s1_df['num_repetitions'] > 0]
        s1_df = s1_df[~s1_df['nsdId'].isin(shared1000_nsdids)]
        s2_df = df_dict[i+1]
        s2_df = s2_df[s2_df['num_repetitions'] > 0]
        s2_df = s2_df[~s2_df['nsdId'].isin(shared1000_nsdids)]
        sbjsbj_stimoverlap[i, j] = len(set(s1_df['cocoId']).intersection(set(s2_df['cocoId'])))
    sbjsbj_stimoverlap[i, i] = 0
print(sbjsbj_stimoverlap)

[[0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0.]]


In [2]:
shared1000_nsdids = nsd_ds_utils.get_subset_nsdids(NSD_DIR, 1000)

sbjsbj_stimoverlap = np.zeros((8, 8))
for i in range(8):
    s1_df = df_dict[i+1]
    s1_df = s1_df[s1_df['num_repetitions'] > 0]
    s1_df = s1_df[~s1_df['nsdId'].isin(shared1000_nsdids)]
    print(f'Subject {i+1} has {len(s1_df)} images after removing shared1000')

NameError: name 'df_dict' is not defined

# Subject-specific images

In [3]:
# subjects = [1, 2, 5, 7]
subjects = [3, 4, 6, 8]
shared1000_nsdids = nsd_ds_utils.get_subset_nsdids(NSD_DIR, 1000)
# Get all stimuli
nsdcoco_stim = nsd.get_stimuli_dataset(
    NSD_DIR, transform=None, subset=None
).all_stimuli

for subject in subjects:
    _, images_df = nsd.get_betas(
        NSD_DIR, subject, 'FFA', 7, subset=None, voxels=False
    )
    images_df = images_df[images_df['num_repetitions'] > 0].reset_index(drop=True)
    # Remove shared1000 images
    images_df = images_df[~images_df['nsdId'].isin(shared1000_nsdids)].reset_index(drop=True)
    images_df = coco_utils.add_image_info_to_df(images_df, paths.COCO_ANNOTATIONS_PATH)
    print(f'Subject {subject} has {len(images_df)} subject-specific images with atleast 1 repetitions')

    images_save_dir = f'{RESULTS_DIR}/all_subject_specific_images/subject_{subject}'
    os.makedirs(images_save_dir, exist_ok=True)
    img_file_idx = 0
    for idx, row in images_df.iterrows():
        img = nsdcoco_stim[row['nsdId']]
        img = Image.fromarray((img).astype(np.uint8))
        img.save(
            f'{images_save_dir}/idx_{img_file_idx}_nsdId_{row["nsdId"]}_cocoId_{row["cocoId"]}.jpg'
        )
        img_file_idx += 1

loading annotations into memory...
Done (t=7.45s)
creating index...
index created!
loading annotations into memory...
Done (t=0.52s)
creating index...
index created!
loading annotations into memory...
Done (t=0.31s)
creating index...
index created!
loading annotations into memory...
Done (t=0.05s)
creating index...
index created!
Subject 3 has 8481 subject-specific images with atleast 1 repetitions
loading annotations into memory...
Done (t=7.05s)
creating index...
index created!
loading annotations into memory...
Done (t=0.52s)
creating index...
index created!
loading annotations into memory...
Done (t=0.30s)
creating index...
index created!
loading annotations into memory...
Done (t=0.02s)
creating index...
index created!
Subject 4 has 8302 subject-specific images with atleast 1 repetitions
loading annotations into memory...
Done (t=7.10s)
creating index...
index created!
loading annotations into memory...
Done (t=0.52s)
creating index...
index created!
loading annotations into memor

# Subject 1

In [ ]:
subject = 1
froi = 'FFA'
tthreshold = 7

_, images_df = nsd.get_betas(
    NSD_DIR, subject, froi, tthreshold, subset=None, voxels=False
)
shared1000_nsdids = nsd_ds_utils.get_subset_nsdids(NSD_DIR, 1000)
# Remove shared1000 images
images_df = images_df[~images_df['nsdId'].isin(shared1000_nsdids)].reset_index(drop=True)
images_df = coco_utils.add_image_info_to_df(images_df)
# Get all stimuli
nsdcoco_stim = nsd.get_stimuli_dataset(
    NSD_DIR, transform=None, subset=None
).all_stimuli
# Find images with people but no animals
person_no_animal_df = images_df[
    images_df['labels'].apply(
        lambda labels: any(label in coco_person_labels for label in labels) and 
            not any(label in coco_animal_labels for label in labels) and 
            not any(label in exclusion_labels for label in labels)
    )
].reset_index(drop=True)
print(f"Number of images with people but no animals: {len(person_no_animal_df)}")
# Find images with animals but no people
animal_no_person_df = images_df[
    images_df['labels'].apply(
        lambda labels: any(label in coco_animal_labels for label in labels) and 
            not any(label in coco_person_labels for label in labels) and 
            not any(label in exclusion_labels for label in labels)
    )
].reset_index(drop=True)
print(f"Number of images with animals but no people: {len(animal_no_person_df)}")

Number of images with people but no animals: 4083
Number of images with animals but no people: 1409


In [6]:
# Save the images with name
images_save_dir = f'{RESULTS_DIR}/subject_{subject}/candidate_ppa_images'
os.makedirs(images_save_dir, exist_ok=True)
img_file_idx = 0
for idx, row in images_df.iterrows():
    img = nsdcoco_stim[row['nsdId']]
    img = Image.fromarray((img).astype(np.uint8))
    img.save(
        f'{images_save_dir}/idx_{img_file_idx}_nsdId_{row["nsdId"]}_cocoId_{row["cocoId"]}.jpg'
    )
    img_file_idx += 1

In [15]:
# Save the images with name
images_save_dir = f'{RESULTS_DIR}/subject_{subject}/candidate_face_images'
os.makedirs(images_save_dir, exist_ok=True)
img_file_idx = 0
for idx, row in person_no_animal_df.iterrows():
    img = nsdcoco_stim[row['nsdId']]
    img = Image.fromarray((img).astype(np.uint8))
    img.save(
        f'{images_save_dir}/idx_{img_file_idx}_nsdId_{row["nsdId"]}_cocoId_{row["cocoId"]}_person.jpg'
    )
    img_file_idx += 1
for idx, row in animal_no_person_df.iterrows():
    img = nsdcoco_stim[row['nsdId']]
    img = Image.fromarray((img).astype(np.uint8))
    img.save(
        f'{images_save_dir}/idx_{img_file_idx}_nsdId_{row["nsdId"]}_cocoId_{row["cocoId"]}_animal.jpg'
    )
    img_file_idx += 1

In [15]:
results_dir = f'{RESULTS_DIR}/subject_{subject}'
os.makedirs(results_dir, exist_ok=True)
# Sort dataframes by CocoId
person_no_animal_df = person_no_animal_df.sort_values(by='cocoId').reset_index(drop=True)
animal_no_person_df = animal_no_person_df.sort_values(by='cocoId').reset_index(drop=True)
# Save dataframes
person_no_animal_df.to_csv(f'{results_dir}/person_no_animal_images_info.csv', index=False)
animal_no_person_df.to_csv(f'{results_dir}/animal_no_person_images_info.csv', index=False)
# Also save another copy for adding annotations
person_no_animal_df.to_csv(f'{results_dir}/person_no_animal_images_info_for_annotation.csv', index=False)
animal_no_person_df.to_csv(f'{results_dir}/animal_no_person_images_info_for_annotation.csv', index=False)

In [ ]:
person_annotations_tmp = [] # person_only, faces_only, single_person

In [ ]:
# Lists to store the images with canonical faces by manually going through images
person_face_nsdids = [
    488, # also shows hands
    1136, # but also shows hands
    1346,
    2125, # also has baseball glove and ball
    2235, # also has hands
    2516, # also has hands
    3033, # also has hands
    3611,
    3915, # also has hands and food
    3920, # also has hands and food
    4201, # also has hands and torso
    4476, # also has hands
    5202, # also has hands and food
    5460, # also has hands
    5703, # also has hands
    5823, # also has hands
    5892, # also has torso
    6504, # also has hands
    6679, # also has hands and food
    6990, # also has hands and food
    7627, # also has hands
    8557, # also has torso and food
    8593, # also has hands
    8743, # also has hands and food
    8956, # also has hands
    10459, # also has hands
    10479, # also has hands
    11246, # also has torso and food
    11517, # also has hands and food
    12013,
    12258,
    12670, # also has hands and food
    13188, # partial face
    13347, # also has hands and torso
    13972, # also has hands and torso
    14270, # black and white, rotated, has torso
    14380, # also has torso
    14631, # also has hands
    15438, # also has hands
    15510, # also has hands and food
    15835,
    15918, # also has hands
    15977, # also has hands
    16649,
    16952, # also has hands
    17587,
    17651, # also has hands and food
    18170,
    18888, # also has torso
    18960, # also has hands and food
    19301,
    19471,
    22234,
    23811,
    25070, # also has hands
    25999, # also has hands and torso
    26953, # also has hands and torso
    28588, # also has hands and food
    28611, # also has hands and torso
    29901, # also has torso
    30016,
    30174, # also has hands and torso
    31375, # also has hands and food
    31394, # also has hands
    31655, # also has hands and food
    34200, # also has hands
    34290, # also has torso
    34317, # also has hands
    34652, # also has hands and food
    35887, # also has hands and torso
    38317, # also has food
    38458, # also has food
    39079, # also has torso
    39302,
    39390, # also has hands and torso
    40325, # also has hands and food
    40530,
    41853, # also has hands
    42195, # also has torso
    42565,
    43042, # also has hands
    43105, # also has hands
    43038, # also has hands
    43822, # also has hands and food
    45454, # also has torso
    46059,
    46807,
    47988, # also has torso
    49796, # also has hands
    50659, # also has hands
    55396, # also has hands
    55545, # also has hands
    55896, # also has hands
    56047, # also has torso
    57811, # also has background
    58182, # also has torso
    58457,
    59180, # also has hands and food
    60496, # also has hands
    63371,
    65222,
    65730,
    66383, # also has torso
    67148, # also has hands and food
    68315, # also has hands and torso
    69924, # partial face
    70414, # also has hands and food
    70765,
    72340,
    72371,
    72745, # also has hands and food

]
animal_face_nsdids = [
    317, # also has hands and torso
    969, # also has hands and torso
    1162,
    1487, # also has hands and torso
    2795,
    3935, # also has hands and torso
    4037, # also has hands and torso
    4719,
    5156,
    5643,
    6137,
    6315,
    6603, # also has hands and torso
    7274, # also has hands and torso
    8213,
    8486,
    8757,
    8858,
    9268,
    10319, # also has hands and torso
    11431,
    13065, # also has hands and torso
    15180,
    15555, # also has hands and torso
    17303,
    18569,
    19079,
    21064,
    23796,
    25131,
    27204, # also has hands and torso
    27863, # also has hands and torso
    29473, # also has hands and torso
    30602, # also has hands and torso
    33084,
    34473,
    36205,
    37457,
    38898,
    39685, # also has hands and torso
    40527, # also has hands and torso
    43369, # also has hands and torso
    44967, # also has hands and torso
    45318,
    45977,
    46791,
    47540,
    49475, # also has hands and torso
    51406,
    52756,
    54696,
    56706,
    60923,
    68633,
    70461,
]